# BI Assignment — Data Cleaning Notebook


## Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from dateutil import parser as dateparser

# If dateutil not installed, run: pip install python-dateutil openpyxl
print('Libraries loaded successfully')

Libraries loaded successfully


## Step 1 — Load the Raw Data

In [ ]:
df = pd.read_excel('BI_Assignment_Dirty_Data.xlsx', sheet_name='SalesData_Raw')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

Shape: (205, 18)
Columns: ['Order_ID', 'order date', 'CUSTOMER_NAME', 'Customer_Email', 'customer_city', 'Region', 'Product Category', 'Product_Name', 'Quantity', 'Unit Price (USD)', 'Total_Sales', 'Discount%', 'Sales_Rep', 'Channel', 'Return_Status', 'Payment_Method', 'Delivery_Days', 'Customer_Rating']


,Order_ID,order date,CUSTOMER_NAME,Customer_Email,customer_city,Region,Product Category,Product_Name,Quantity,Unit Price (USD),Total_Sales,Discount%,Sales_Rep,Channel,Return_Status,Payment_Method,Delivery_Days,Customer_Rating
0,ORD-1001,NaN,Sara Iqbal,invalid-email,faisalabad,east,Clothing,Sneakers,1.0,5500,380400.0,25,Zara Qureshi,in-store,NaN,Debit Card,1,2.0
1,ORD-1002,12-15-2024,Bilal Hussain,@nodomain.com,Lahore,North,Furniture,Chair,12.0,12000,286137.0,0,Zara Qureshi,In-Store,0,Debit Card,10,3.0
2,ORD-1003,2024-08-14,Fatima Noor,fatima@gmail.com,LAHORE,North,CLOTHING,T-Shirt,15.0,800,12000.0,10,TBD,ONLINE,Yes,credit card,4,4.0
3,ORD-1004,2023-10-04,Sara Iqbal,saraiq@gmail.com,Karachi,south,Electronics,Tablet,3.0,$35000,468828.0,25,Usman Malik,online,YES,NaN,12,5.0
4,ORD-1005,13/02/2024,Anonymous,anonym@gmail.com,Karachi,North,electronics,Tablet,14.0,35000,490000.0,20,NaN,ONLINE,0,Credit Card,2,5.0


## Step 2 — Rename Columns

In [ ]:
df.columns = [
    'Order_ID', 'Order_Date', 'Customer_Name', 'Customer_Email',
    'Customer_City', 'Region', 'Product_Category', 'Product_Name',
    'Quantity', 'Unit_Price', 'Total_Sales', 'Discount_Pct',
    'Sales_Rep', 'Channel', 'Return_Status', 'Payment_Method',
    'Delivery_Days', 'Customer_Rating'
]

print('Columns renamed:')
print(df.columns.tolist())

Columns renamed:
['Order_ID', 'Order_Date', 'Customer_Name', 'Customer_Email', 'Customer_City', 'Region', 'Product_Category', 'Product_Name', 'Quantity', 'Unit_Price', 'Total_Sales', 'Discount_Pct', 'Sales_Rep', 'Channel', 'Return_Status', 'Payment_Method', 'Delivery_Days', 'Customer_Rating']


## Step 3 — Remove Blank Rows & Duplicates

In [ ]:
before = len(df)

df.dropna(how='all', inplace=True)       # remove fully blank rows
df.reset_index(drop=True, inplace=True)

print(f'Rows before: {before}')
print(f'Rows after:  {len(df)}')
print(f'Removed:     {before - len(df)} rows')

Rows before: 205
Rows after:  205
Removed:     0 rows


## Step 4 — Replace Placeholder Text with NaN

In [ ]:
placeholders = ['TBD', 'tbd', 'N/A', 'n/a', 'NA', 'na',
                'missing', 'Missing', 'pending', 'Pending',
                'Anonymous', 'anonymous', '-', '']

for col in df.columns:
    df[col] = df[col].replace(placeholders, np.nan)

print('Null counts after placeholder replacement:')
print(df.isnull().sum())

Null counts after placeholder replacement:
Order_ID             0
Order_Date          32
Customer_Name       33
Customer_Email      29
Customer_City        0
Region               0
Product_Category     0
Product_Name         0
Quantity             9
Unit_Price          10
Total_Sales         23
Discount_Pct        16
Sales_Rep           60
Channel             22
Return_Status       46
Payment_Method      45
Delivery_Days       11
Customer_Rating      8
dtype: int64


/tmp/ipykernel_40270/1714221781.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(placeholders, np.nan)


## Step 5 — Fix Inconsistent Casing

In [ ]:
casing_cols = ['Region', 'Channel', 'Product_Category',
               'Payment_Method', 'Customer_City', 'Customer_Name']

for col in casing_cols:
    df[col] = df[col].str.strip().str.title()

# Fix channel values after title case
df['Channel'] = df['Channel'].replace({
    'In-Store': 'In-Store',
    'Instore':  'In-Store',
    'In Store': 'In-Store'
})

print('Region values:  ', df['Region'].unique())
print('Channel values: ', df['Channel'].unique())
print('Category values:', df['Product_Category'].unique())

Region values:   ['East' 'North' 'South' 'Central' 'West']
Channel values:  ['In-Store' 'Online' 'Retail' nan]
Category values: ['Clothing' 'Furniture' 'Electronics' 'Sports' 'Books' 'Food & Beverage']


## Step 6 — Parse Mixed Date Formats

In [ ]:
def parse_date(val):
    """Handles: yyyy-mm-dd, mm-dd-yyyy, dd/mm/yyyy, dd-Mon-yyyy, dd-mm-yyyy"""
    if pd.isna(val) or str(val).strip() in ['TBD', 'tbd', '', 'nan']:
        return pd.NaT
    val = str(val).strip()
    formats = [
        '%m-%d-%Y',   # 12-15-2024  US format — must be first
        '%Y-%m-%d',   # 2024-08-14  ISO
        '%d/%m/%Y',   # 13/02/2024  UK slashes
        '%m/%d/%Y',   # 02/13/2024  US slashes
        '%d-%b-%Y',   # 01-Jan-2023 month name
        '%d-%m-%Y',   # 01-02-2024  ambiguous fallback
        '%Y/%m/%d',   # 2024/08/14  ISO slashes
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            continue
    try:
        return dateparser.parse(val, dayfirst=False)
    except:
        return pd.NaT

df['Order_Date'] = df['Order_Date'].apply(parse_date)

print('Date dtype:', df['Order_Date'].dtype)
print('Null dates:', df['Order_Date'].isna().sum())
df['Order_Date'].head(10)

Date dtype: datetime64[ns]
Null dates: 32


,Order_Date
0,NaT
1,2024-12-15
2,2024-08-14
3,2023-10-04
4,2024-02-13
5,2023-10-01
6,2023-01-01
7,2024-12-07
8,2023-09-01
9,2023-07-16


## Step 7 — Clean Unit_Price: Remove Currency Symbols

In [ ]:
df['Unit_Price'] = (
    df['Unit_Price']
    .astype(str)
    .str.replace(r'[\$,PKR\s]', '', regex=True)
    .str.strip()
)
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')
df['Unit_Price'] = df['Unit_Price'].abs()   # fix negatives (Issue 7)

print('Unit_Price sample:')
print(df['Unit_Price'].head(10).tolist())
print('Nulls:', df['Unit_Price'].isna().sum())

Unit_Price sample:
[5500.0, 12000.0, 800.0, 35000.0, 35000.0, 3500.0, 45000.0, 25000.0, 3200.0, 800.0]
Nulls: 10


## Step 8 — Fix Negative Quantities

In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').abs()

print('Quantity sample:')
print(df['Quantity'].describe())

Quantity sample:
count    196.000000
mean      10.301020
std        6.065979
min        1.000000
25%        5.000000
50%       10.000000
75%       16.000000
max       20.000000
Name: Quantity, dtype: float64


## Step 9 — Fix Discount_Pct

In [ ]:
# Remove % sign and convert to number
df['Discount_Pct'] = (
    df['Discount_Pct']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.strip()
)
df['Discount_Pct'] = pd.to_numeric(df['Discount_Pct'], errors='coerce')

# Convert decimal format (0.25 → 25)
df['Discount_Pct'] = df['Discount_Pct'].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x < 1 else x
)

# Null out impossible values (negative or above 100)
df['Discount_Pct'] = df['Discount_Pct'].apply(
    lambda x: np.nan if pd.notna(x) and (x < 0 or x > 100) else x
)

print('Discount_Pct range:', df['Discount_Pct'].min(), 'to', df['Discount_Pct'].max())
print('Nulls:', df['Discount_Pct'].isna().sum())

Discount_Pct range: 0.0 to 30.0
Nulls: 27


## Step 10 — Recalculate Total_Sales

In [ ]:
df['Total_Sales_Correct'] = (
    df['Quantity']
    * df['Unit_Price']
    * (1 - df['Discount_Pct'].fillna(0) / 100)
).round(2)

# Flag rows where original Total_Sales was wrong
df['Total_Mismatch_Flag'] = (
    (df['Total_Sales'] - df['Total_Sales_Correct']).abs() > 1
)

print('Rows with wrong Total_Sales:', df['Total_Mismatch_Flag'].sum())
df[df['Total_Mismatch_Flag']][['Order_ID','Quantity','Unit_Price',
                                'Discount_Pct','Total_Sales',
                                'Total_Sales_Correct']].head(10)

Rows with wrong Total_Sales: 149


,Order_ID,Quantity,Unit_Price,Discount_Pct,Total_Sales,Total_Sales_Correct
0,ORD-1001,1.0,5500.0,25.0,380400.0,4125.0
1,ORD-1002,12.0,12000.0,0.0,286137.0,144000.0
2,ORD-1003,15.0,800.0,10.0,12000.0,10800.0
3,ORD-1004,3.0,35000.0,25.0,468828.0,78750.0
4,ORD-1005,14.0,35000.0,20.0,490000.0,392000.0
5,ORD-1006,6.0,3500.0,25.0,21000.0,15750.0
6,ORD-1007,3.0,45000.0,5.0,70934.0,128250.0
7,ORD-1008,8.0,25000.0,20.0,200000.0,160000.0
8,ORD-1009,7.0,3200.0,0.0,387245.0,22400.0
9,ORD-1010,18.0,800.0,NaN,449960.0,14400.0


## Step 11 — Standardize Return_Status

In [ ]:
ret_map = {
    'yes': 'Yes', 'YES': 'Yes', '1': 'Yes', 1: 'Yes', 1.0: 'Yes',
    'no':  'No',  'NO':  'No',  '0': 'No',  0: 'No',  0.0: 'No'
}
df['Return_Status'] = df['Return_Status'].replace(ret_map)
df['Return_Status'] = df['Return_Status'].apply(
    lambda x: x if x in ['Yes', 'No'] else np.nan
)

print('Return_Status values:', df['Return_Status'].value_counts())

Return_Status values: Return_Status
No     82
Yes    77
Name: count, dtype: int64


## Step 12 — Fix Delivery_Days Outliers

In [ ]:
df['Delivery_Days'] = pd.to_numeric(df['Delivery_Days'], errors='coerce')
df['Delivery_Days'] = df['Delivery_Days'].apply(
    lambda x: np.nan if pd.notna(x) and (x <= 0 or x > 90) else x
)

print('Delivery_Days range:', df['Delivery_Days'].min(), 'to', df['Delivery_Days'].max())
print('Nulls:', df['Delivery_Days'].isna().sum())

Delivery_Days range: 1.0 to 14.0
Nulls: 24


## Step 13 — Fix Customer_Rating Out-of-Range

In [ ]:
df['Customer_Rating'] = pd.to_numeric(df['Customer_Rating'], errors='coerce')
df['Customer_Rating'] = df['Customer_Rating'].apply(
    lambda x: np.nan if pd.notna(x) and (x < 1 or x > 5) else x
)

print('Rating values:', df['Customer_Rating'].value_counts().sort_index())

Rating values: Customer_Rating
1.0    32
2.0    35
3.0    35
3.5     3
4.0    42
5.0    37
Name: count, dtype: int64


## Step 14 — Flag Invalid Emails

In [ ]:
def validate_email(e):
    if pd.isna(e):
        return 'Missing'
    e = str(e).strip()
    pattern = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'
    if re.match(pattern, e) and not e.startswith('@'):
        return 'Valid'
    return 'Invalid'

df['Email_Flag'] = df['Customer_Email'].apply(validate_email)

print('Email flags:', df['Email_Flag'].value_counts())

Email flags: Email_Flag
Valid      144
Invalid     32
Missing     29
Name: count, dtype: int64


## Step 15 — Final Summary: Before vs After

In [ ]:
print('='*50)
print('CLEANING SUMMARY')
print('='*50)
print(f'Total rows (cleaned):        {len(df)}')
print(f'Total columns:               {len(df.columns)}')
print(f'Wrong Total_Sales fixed:     {df["Total_Mismatch_Flag"].sum()} rows')
print(f'Invalid emails flagged:      {(df["Email_Flag"]=="Invalid").sum()} rows')
print(f'Missing emails:              {(df["Email_Flag"]=="Missing").sum()} rows')
print()
print('Remaining nulls per column:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)
df.head()

CLEANING SUMMARY
Total rows (cleaned):        205
Total columns:               21
Wrong Total_Sales fixed:     149 rows
Invalid emails flagged:      32 rows
Missing emails:              29 rows

Remaining nulls per column:
Order_ID                0
Order_Date             32
Customer_Name          33
Customer_Email         29
Customer_City           0
Region                  0
Product_Category        0
Product_Name            0
Quantity                9
Unit_Price             10
Total_Sales            23
Discount_Pct           27
Sales_Rep              60
Channel                22
Return_Status          46
Payment_Method         45
Delivery_Days          24
Customer_Rating        21
Total_Sales_Correct    19
Total_Mismatch_Flag     0
Email_Flag              0
dtype: int64

Data types:
Order_ID                       object
Order_Date             datetime64[ns]
Customer_Name                  object
Customer_Email                 object
Customer_City                  object
Region         

,Order_ID,Order_Date,Customer_Name,Customer_Email,Customer_City,Region,Product_Category,Product_Name,Quantity,Unit_Price,...,Discount_Pct,Sales_Rep,Channel,Return_Status,Payment_Method,Delivery_Days,Customer_Rating,Total_Sales_Correct,Total_Mismatch_Flag,Email_Flag
0,ORD-1001,NaT,Sara Iqbal,invalid-email,Faisalabad,East,Clothing,Sneakers,1.0,5500.0,...,25.0,Zara Qureshi,In-Store,NaN,Debit Card,1.0,2.0,4125.0,True,Invalid
1,ORD-1002,2024-12-15,Bilal Hussain,@nodomain.com,Lahore,North,Furniture,Chair,12.0,12000.0,...,0.0,Zara Qureshi,In-Store,No,Debit Card,10.0,3.0,144000.0,True,Invalid
2,ORD-1003,2024-08-14,Fatima Noor,fatima@gmail.com,Lahore,North,Clothing,T-Shirt,15.0,800.0,...,10.0,NaN,Online,Yes,Credit Card,4.0,4.0,10800.0,True,Valid
3,ORD-1004,2023-10-04,Sara Iqbal,saraiq@gmail.com,Karachi,South,Electronics,Tablet,3.0,35000.0,...,25.0,Usman Malik,Online,Yes,NaN,12.0,5.0,78750.0,True,Valid
4,ORD-1005,2024-02-13,NaN,anonym@gmail.com,Karachi,North,Electronics,Tablet,14.0,35000.0,...,20.0,NaN,Online,No,Credit Card,2.0,5.0,392000.0,True,Valid


## Step 16 — Export Cleaned File

In [ ]:
output_path = 'BI_Cleaned_Data.xlsx'

df.to_excel(output_path, index=False, sheet_name='Cleaned_Data')

print(f'Saved to: {output_path}')
print('Ready to import into Power BI!')

Saved to: BI_Cleaned_Data.xlsx
Ready to import into Power BI!
